# Polyhedral Template Matching (PTM)

Polyhedral Template Matching (Larsen, Schmidt & Schiøtz, *MSMSE* **24**,
055007, 2016) identifies local crystal structures by matching each
atom's neighbourhood against ideal polyhedral templates.

**Recognised structures:** FCC, HCP, BCC, ICO, SC, diamond cubic,
diamond hexagonal, graphene.

PTM is **more robust than CNA** at finite temperature because it
performs an optimal rotation fit rather than relying on fixed cutoffs.
It also extracts orientations (quaternions) and interatomic distances.

In [1]:
import numpy as np
import pyscal3 as pc
from ase.build import bulk

## 1. Perfect FCC

In [2]:
al = bulk("Al", cubic=True).repeat(3)
types = pc.polyhedral_template_matching(al)

unique, counts = np.unique(types, return_counts=True)
for t, c in zip(unique, counts):
    label = pc.PTM_TYPES[t]
    print("{:>10s}: {} atoms".format(label, c))

print("\nMean RMSD:", al.arrays["pyscal_ptm_rmsd"].mean())
print("Interatomic distance:", al.arrays["pyscal_ptm_interatomic_distance"][0])

       fcc: 108 atoms

Mean RMSD: 2.197454128547333e-08
Interatomic distance: 2.8637824638055176


## 2. Comparing FCC, BCC, HCP

In [3]:
structures = {
    "FCC Al": bulk("Al", cubic=True).repeat(3),
    "BCC Fe": bulk("Fe", cubic=True).repeat(3),
    "HCP Mg": bulk("Mg", "hcp", a=3.21, c=5.21).repeat((3, 3, 3)),
}

for name, atoms in structures.items():
    types = pc.polyhedral_template_matching(atoms)
    unique, counts = np.unique(types, return_counts=True)
    labels = [pc.PTM_TYPES[t] for t in unique]
    rmsd = atoms.arrays["pyscal_ptm_rmsd"].mean()
    print("{}: {} (mean RMSD = {:.2e})".format(name, dict(zip(labels, counts)), rmsd))

FCC Al: {'fcc': np.int64(108)} (mean RMSD = 2.20e-08)
BCC Fe: {'bcc': np.int64(54)} (mean RMSD = 1.20e-08)
HCP Mg: {'hcp': np.int64(54)} (mean RMSD = 2.76e-03)


## 3. Diamond and Simple Cubic

In [4]:
si = bulk("Si", "diamond", cubic=True).repeat(2)
types = pc.polyhedral_template_matching(si, structures="all")
unique, counts = np.unique(types, return_counts=True)
labels = [pc.PTM_TYPES[t] for t in unique]
print("Diamond Si:", dict(zip(labels, counts)))

po = bulk("Po", "sc", a=3.35).repeat(3)
types = pc.polyhedral_template_matching(po, structures="all")
unique, counts = np.unique(types, return_counts=True)
labels = [pc.PTM_TYPES[t] for t in unique]
print("SC Po:     ", dict(zip(labels, counts)))

Diamond Si: {'dcub': np.int64(64)}
SC Po:      {'sc': np.int64(27)}


## 4. Thermal noise robustness

In [5]:
al = bulk("Al", cubic=True).repeat(4)
rng = np.random.default_rng(42)

for sigma in [0.0, 0.02, 0.05, 0.10, 0.15]:
    atoms = al.copy()
    atoms.positions += rng.normal(0, sigma, atoms.positions.shape)
    types = pc.polyhedral_template_matching(atoms, rmsd_cutoff=0.15)
    frac_fcc = np.mean(types == 1)
    mean_rmsd = atoms.arrays["pyscal_ptm_rmsd"]
    mean_rmsd = mean_rmsd[mean_rmsd < np.inf].mean()
    print("sigma={:.2f}: {:.1%} FCC, mean RMSD={:.4f}".format(
        sigma, frac_fcc, mean_rmsd))

sigma=0.00: 100.0% FCC, mean RMSD=0.0000
sigma=0.02: 100.0% FCC, mean RMSD=0.0112
sigma=0.05: 100.0% FCC, mean RMSD=0.0270
sigma=0.10: 100.0% FCC, mean RMSD=0.0557
sigma=0.15: 100.0% FCC, mean RMSD=0.0784


## 5. Crystal orientation

PTM also extracts a quaternion representing the orientation of each
atom's neighbourhood relative to the ideal template.

In [6]:
al = bulk("Al", cubic=True).repeat(2)
pc.polyhedral_template_matching(al)
q = al.arrays["pyscal_ptm_orientation"]
print("Orientation quaternions (first 3 atoms):")
print(q[:3])
print("\nQuaternion norms:", np.linalg.norm(q[:3], axis=1))

Orientation quaternions (first 3 atoms):
[[ 1.00000000e+00 -1.39093002e-17 -1.01465364e-17  1.01421538e-19]
 [ 1.00000000e+00  2.45947189e-36  6.85322855e-18 -8.96191425e-18]
 [ 1.00000000e+00  2.77555756e-17  2.77555756e-17  2.77555756e-17]]

Quaternion norms: [1. 1. 1.]


## Summary

| Feature | CNA | PTM |
|---------|-----|-----|
| Noise robustness | Moderate | **High** |
| Orientation | No | **Yes (quaternion)** |
| Structures | FCC/BCC/HCP | **8 types** |
| Speed | Fast | Fast (C++ library) |

PTM is recommended over CNA for finite-temperature simulations
and grain-boundary analysis where orientation information is needed.